# Telco Customer Churn — Machine Learning Comparison

**Goal:** Predict whether a telecommunications customer will churn, comparing Logistic Regression, Random Forest, and a Multi-Layer Perceptron (MLP).

**Attribution:** Adapted from the student-provided Kaggle notebook `customer-churn-prediction.ipynb` (original author and license should be credited after verification). Dataset: https://www.kaggle.com/datasets/blastchar/telco-customer-churn. Model scores are calculated when the notebook is run.

## 1. Imports and dataset loading
Download `WA_Fn-UseC_-Telco-Customer-Churn.csv` into `data/`. In Kaggle, the notebook also searches `/kaggle/input/`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay)

SEED = 42
name = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
candidates = [Path(name), Path('data') / name, Path('../data') / name]
candidates += list(Path('/kaggle/input').glob('**/' + name)) if Path('/kaggle/input').exists() else []
path = next((p for p in candidates if p.exists()), None)
if path is None:
    raise FileNotFoundError('Деректер жоқ: Kaggle Telco Customer Churn CSV файлын жүктеңіз (README қараңыз).')
df = pd.read_csv(path)
print('Деректер өлшемі:', df.shape)
display(df.head())

## 2. Cleaning and exploratory analysis
Convert `TotalCharges` to numeric and handle missing values inside the preprocessing pipeline. Exclude `customerID` from training.

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print('Missing values:', df.isna().sum()[df.isna().sum() > 0].to_dict())
print('Churn distribution (%):')
print((df['Churn'].value_counts(normalize=True)*100).round(2))
ax = df['Churn'].value_counts().reindex(['No', 'Yes']).plot(kind='bar', rot=0, title='Customer churn distribution')
ax.set_xlabel('Churn'); ax.set_ylabel('Number of customers'); plt.tight_layout(); plt.show()

## 3. Train/test split and preprocessing
Apply a stratified train/test split. Fit encoders and scalers only on the training portion.

In [ ]:
X = df.drop(columns=['Churn', 'customerID'], errors='ignore')
y = df['Churn'].map({'No': 0, 'Yes': 1})
if y.isna().any():
    raise ValueError('Churn column must contain Yes/No only.')
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)
# sklearn ескі және жаңа нұсқаларында жұмыс істейді
try:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)
preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')),
                          ('scaler', StandardScaler())]), num_cols),
    ('category', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                           ('encoder', encoder)]), cat_cols)
])
print('Train:', X_train.shape, '| Test:', X_test.shape)

## 4. Train three models
Keep the original neural network idea as `MLPClassifier` and compare it with Logistic Regression and Random Forest.

In [ ]:
algorithms = {
    'Logistic Regression': LogisticRegression(max_iter=1500, random_state=SEED),
    'Random Forest': RandomForestClassifier(n_estimators=250, min_samples_leaf=3,
                                            random_state=SEED, n_jobs=-1),
    'Neural Network (MLP)': MLPClassifier(hidden_layer_sizes=(32, 16),
                                         early_stopping=True, max_iter=250,
                                         random_state=SEED)
}
models = {}
records = []
for name, estimator in algorithms.items():
    pipeline = Pipeline([('preprocess', preprocessor), ('model', estimator)])
    pipeline.fit(X_train, y_train)
    prediction = pipeline.predict(X_test)
    probability = pipeline.predict_proba(X_test)[:, 1]
    records.append({
        'model': name,
        'accuracy': accuracy_score(y_test, prediction),
        'precision': precision_score(y_test, prediction, zero_division=0),
        'recall': recall_score(y_test, prediction, zero_division=0),
        'f1': f1_score(y_test, prediction, zero_division=0),
        'roc_auc': roc_auc_score(y_test, probability)
    })
    models[name] = pipeline
metrics = pd.DataFrame(records).sort_values('roc_auc', ascending=False)
display(metrics.style.format({col: '{:.3f}' for col in ['accuracy','precision','recall','f1','roc_auc']}))

## 5. Compare evaluation metrics

In [ ]:
ax = metrics.set_index('model')[['accuracy','f1','roc_auc']].plot(kind='bar', figsize=(9,5), rot=15)
ax.set_ylabel('Metric (0–1)'); ax.set_title('Model comparison'); ax.set_ylim(0,1)
plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(7,5))
for name, pipeline in models.items():
    RocCurveDisplay.from_estimator(pipeline, X_test, y_test, ax=ax, name=name)
ax.set_title('ROC қисықтары'); plt.tight_layout(); plt.show()

## 6. Best ROC-AUC model and confusion matrix

In [ ]:
best_name = metrics.iloc[0]['model']
best_pipeline = models[best_name]
ConfusionMatrixDisplay.from_estimator(best_pipeline, X_test, y_test, display_labels=['Stayed', 'Churned'], cmap='Blues')
plt.title('Confusion matrix: ' + best_name); plt.tight_layout(); plt.show()
print('Best model by ROC-AUC:', best_name)
print('ROC-AUC = {:.2f}%, Accuracy = {:.2f}%, F1 = {:.2f}%'.format(
    metrics.iloc[0]['roc_auc']*100, metrics.iloc[0]['accuracy']*100, metrics.iloc[0]['f1']*100))
print('Үздік және ең төмен ROC-AUC айырмасы (пайыздық пункт): {:.2f}'.format(
    (metrics['roc_auc'].max()-metrics['roc_auc'].min())*100))

## 7. Export the results
Save measured metrics to CSV. Do not fill the conclusion with invented percentages.

In [ ]:
from pathlib import Path
output = Path('results')
output.mkdir(exist_ok=True)
metrics.to_csv(output / 'metrics.csv', index=False)
print('Saved:', output / 'metrics.csv')

## Conclusion
This adapted educational project compares Logistic Regression, Random Forest, and an MLP neural network for Telco customer churn prediction. After execution, compare Accuracy, Precision, Recall, F1 and ROC-AUC using the resulting table. This is a single-machine scikit-learn demonstration, not a distributed Apache Spark benchmark.